# 2. 5分割ファイルの作成

## 2.1 ライブラリのインポート

In [5]:
import os
import numpy as np
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import csv
import re

## 2.2 変数宣言、データの読み込み

In [ ]:
model_name = "prott5"
seed = 42

data_dir = f"./data/embedding-vectors_exclusion/{model_name}/base"  # 1024次元の埋め込みベクトルの読み込み

## 2.3 ラベルの読み込み

In [7]:
with open("gpcr_labels_exclusion.csv") as f:
    reader = csv.reader(f)
    l = [row for row in reader]

labels = []
for i in range(len(l)):
    labels.append(l[i][1])

## 2.4 ファイル一覧の取得

In [8]:
files = [os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith(".npy")]  # .npy ファイルを全て取得

sorted_files = sorted(files, key=lambda p: int(re.search(r'(\d+).npy$', p).group(1)))
print(sorted_files)

['./data/embedding-vectors_exclusion/protbert/base\\1.npy', './data/embedding-vectors_exclusion/protbert/base\\2.npy', './data/embedding-vectors_exclusion/protbert/base\\3.npy', './data/embedding-vectors_exclusion/protbert/base\\4.npy', './data/embedding-vectors_exclusion/protbert/base\\5.npy', './data/embedding-vectors_exclusion/protbert/base\\6.npy', './data/embedding-vectors_exclusion/protbert/base\\7.npy', './data/embedding-vectors_exclusion/protbert/base\\8.npy', './data/embedding-vectors_exclusion/protbert/base\\9.npy', './data/embedding-vectors_exclusion/protbert/base\\10.npy', './data/embedding-vectors_exclusion/protbert/base\\11.npy', './data/embedding-vectors_exclusion/protbert/base\\12.npy', './data/embedding-vectors_exclusion/protbert/base\\13.npy', './data/embedding-vectors_exclusion/protbert/base\\14.npy', './data/embedding-vectors_exclusion/protbert/base\\15.npy', './data/embedding-vectors_exclusion/protbert/base\\16.npy', './data/embedding-vectors_exclusion/protbert/bas

## 2.5 層化5分割によるラベルの分割

In [9]:
le = LabelEncoder()
y = le.fit_transform(labels)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
for train_idx, test_idx in skf.split(sorted_files, y):
    print("train_index:", train_idx, "test_index:", test_idx)
    print(train_idx.shape, test_idx.shape)

train_index: [   0    1    3 ... 7609 7610 7611] test_index: [   2   10   11 ... 7601 7603 7606]
(6089,) (1523,)
train_index: [   0    2    3 ... 7607 7609 7611] test_index: [   1    9   13 ... 7602 7608 7610]
(6089,) (1523,)
train_index: [   1    2    5 ... 7609 7610 7611] test_index: [   0    3    4 ... 7597 7599 7605]
(6090,) (1522,)
train_index: [   0    1    2 ... 7609 7610 7611] test_index: [   7   17   19 ... 7595 7598 7604]
(6090,) (1522,)
train_index: [   0    1    2 ... 7606 7608 7610] test_index: [   5    6    8 ... 7607 7609 7611]
(6090,) (1522,)


## 2.6 5分割データの作成

In [10]:
for fold, (train_idx, test_idx) in enumerate(skf.split(sorted_files, y)):
    train_dir = f"./data/embedding-vectors_exclusion/{model_name}/base/fold{fold + 1}_train"
    test_dir = f"./data/embedding-vectors_exclusion/{model_name}/base/fold{fold + 1}_test"
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    idx = 0

    for filename in tqdm(sorted_files):
        arr = np.load(filename)
        save_filename = os.path.basename(filename)
        if idx not in test_idx: #train_idxに入っているならば
            np.save(os.path.join(train_dir, save_filename), arr)
        else:
            np.save(os.path.join(test_dir, save_filename), arr)
        idx += 1

  0%|          | 0/7612 [00:00<?, ?it/s]

100%|██████████| 7612/7612 [09:31<00:00, 13.32it/s]  
